In [0]:
%run ../functions/functions

In [0]:
database_name = "dimensao"
table_name = "dm_ncm"
target_path = f"{database_name}.{table_name}"
pk = "SK_NCM"

In [0]:
silver_path_s = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/NCM/"
silver_path_sh = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/NCM_SH_CONSOLIDADA/"

In [0]:
df_s = spark.read.format("delta").load(silver_path_s)
df_sh = spark.read.format("delta").load(silver_path_sh)

In [0]:
df_s.createOrReplaceTempView("df_ncm")
df_sh.createOrReplaceTempView("df_ncm_sh")

In [0]:
query = """
select 
  c.SK_NCM,
  c.CO_NCM,
  c.NO_NCM_POR,
  s.CO_SH4,
  c.CO_SH6
from df_ncm as c
left join df_ncm_sh as s on try_cast(c.CO_SH6 as BIGINT) = try_cast(s.CO_SH6 as BIGINT)"""

In [0]:
df_final = spark.sql(query)

In [0]:
save_hive_table(df_final, target_path, pk)